In [2]:
import pandas as pd

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------

news_path = "/Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/data/raw_news.csv"
prices_path = "/Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/data/raw_prices.csv"

news = pd.read_csv(news_path)
prices = pd.read_csv(prices_path)

print("Raw news shape:", news.shape)
print("Raw prices shape:", prices.shape)
print("News columns:", news.columns.tolist())
print("Prices columns:", prices.columns.tolist())

# --------------------------------------------------
# 2. DROP UNUSED INDEX COLUMN AND FILTER TICKERS
# --------------------------------------------------

# Your news columns: ['Unnamed: 0', 'title', 'date', 'stock']
# 'Unnamed: 0' is just an old index, we can drop it
if "Unnamed: 0" in news.columns:
    news = news.drop(columns=["Unnamed: 0"])

print("\nAfter dropping Unnamed: 0, news columns:", news.columns.tolist())

# Keep only your 10 tickers
valid_tickers = ['MRK', 'MS', 'MU', 'NVDA', 'QQQ', 'M', 'EBAY', 'NFLX', 'GILD', 'VZ']
news = news[news["stock"].isin(valid_tickers)].copy()

print("News after filtering to 10 tickers:", news.shape)
print("Unique news tickers:", news["stock"].unique())

# --------------------------------------------------
# 3. PARSE DATES AND MAKE DATE-ONLY COLUMNS
# --------------------------------------------------

# Here we FORCE using the real 'date' column, not Unnamed: 0
# Use utc=True to handle timezone warnings
news["date"] = pd.to_datetime(news["date"], errors="coerce", utc=True)
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce", utc=True)

print("\nDtypes after datetime conversion:")
print(news.dtypes)
print(prices.dtypes)

# Date-only versions (no time of day)
news["NewsDate"] = news["date"].dt.date
prices["TradeDate"] = prices["Date"].dt.date

print("\nSample news dates:")
print(news[["date", "NewsDate"]].head())
print("\nSample price dates:")
print(prices[["Date", "TradeDate"]].head())

# --------------------------------------------------
# 4. CREATE NEXT-DAY CLOSE + UP/DOWN LABEL IN PRICES
# --------------------------------------------------

# Sort so next-day shift works per ticker
prices = prices.sort_values(["Ticker", "TradeDate"])

# Next day's close
prices["NextClose"] = prices.groupby("Ticker")["ClosePrice"].shift(-1)

# One-day return
prices["Return1D"] = (prices["NextClose"] - prices["ClosePrice"]) / prices["ClosePrice"]

# Label: 1 if stock goes up next day, 0 otherwise
prices["UpDownLabel"] = (prices["Return1D"] > 0).astype(int)

# Remove rows with no NextClose (last day of each ticker)
prices = prices.dropna(subset=["NextClose"])

print("\nPrices with labels (first few rows):")
print(prices[["Ticker", "TradeDate", "ClosePrice", "NextClose", "Return1D", "UpDownLabel"]].head())
print("Prices after adding labels:", prices.shape)

# --------------------------------------------------
# 5. SELECT PRICE COLUMNS FOR MERGE
# --------------------------------------------------

price_cols_for_merge = [
    "Ticker",
    "TradeDate",
    "ClosePrice",
    "NextClose",
    "Return1D",
    "UpDownLabel"
]

prices_for_merge = prices[price_cols_for_merge].copy()

print("\nPrice subset for merge:")
print(prices_for_merge.head())

# --------------------------------------------------
# 6. MERGE NEWS WITH PRICES
# --------------------------------------------------

merged = pd.merge(
    news,
    prices_for_merge,
    left_on=["stock", "NewsDate"],    # from news
    right_on=["Ticker", "TradeDate"], # from prices
    how="inner"
)

print("\nMerged shape:", merged.shape)
print(merged.head())

# --------------------------------------------------
# 7. CHECK LABEL DISTRIBUTION AND SAVE
# --------------------------------------------------

print("\nLabel counts (0 = not up, 1 = up):")
print(merged["UpDownLabel"].value_counts(dropna=False))

print("\nAverage one day return on merged rows:")
print(merged["Return1D"].mean())

save_path = "/Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/data/merged_news_prices.csv"
merged.to_csv(save_path, index=False)

print("\nSaved merged modeling data to:", save_path)


Raw news shape: (1400469, 4)
Raw prices shape: (28490, 10)
News columns: ['Unnamed: 0', 'title', 'date', 'stock']
Prices columns: ['Date', 'OpenPrice', 'HighPrice', 'LowPrice', 'ClosePrice', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Capital Gains']

After dropping Unnamed: 0, news columns: ['title', 'date', 'stock']
News after filtering to 10 tickers: (30967, 3)
Unique news tickers: ['EBAY' 'GILD' 'M' 'MRK' 'MS' 'MU' 'NFLX' 'NVDA' 'QQQ' 'VZ']

Dtypes after datetime conversion:
title                 object
date     datetime64[ns, UTC]
stock                 object
dtype: object
Date             datetime64[ns, UTC]
OpenPrice                    float64
HighPrice                    float64
LowPrice                     float64
ClosePrice                   float64
Volume                         int64
Dividends                    float64
Stock Splits                 float64
Ticker                        object
Capital Gains                float64
dtype: object

Sample news dates:
     